# 03 — Interactive Inference Demo

Use the fine-tuned Mistral-7B summarizer interactively in this notebook.

**Prerequisites:** Trained adapter at `models/mistral7b-summarizer-qlora/`

In [ ]:
import sys, os
sys.path.insert(0, '.')

# Try Unsloth first (GPU), fall back to PEFT
ADAPTER_PATH = 'models/mistral7b-summarizer-qlora'

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_PATH,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print('✅ Model loaded via Unsloth')
except ImportError:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(
        'mistralai/Mistral-7B-Instruct-v0.3',
        device_map='auto', trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print('✅ Model loaded via PEFT')

In [ ]:
import torch
from src.data.preprocess import INFERENCE_TEMPLATE

def summarize(text, max_new_tokens=200, temperature=0.1):
    """Generate a summary for the given paper text."""
    prompt = INFERENCE_TEMPLATE.format(source=text.strip())
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.01,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs['input_ids'].shape[1]
    decoded = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    return decoded.removeprefix('Summary:').strip()

print('summarize() function ready!')

## Try It Out

In [ ]:
paper_text = """
We present a new approach to fine-tuning large language models using quantized low-rank 
adaptation (QLoRA). QLoRA backpropagates gradients through a frozen, 4-bit quantized 
pretrained language model into Low Rank Adapters. Our best models family, which we name 
Guanaco, outperforms all previous openly released models on the Vicuna benchmark, reaching 
99.3% of the performance level of ChatGPT while only requiring 24 hours of finetuning on 
a single GPU. QLoRA introduces a number of innovations to save memory without sacrificing 
performance: (a) 4-bit NormalFloat (NF4), a new data type that is information theoretically 
optimal for normally distributed weights (b) double quantization to reduce the average memory 
footprint by quantizing the quantization constants, and (c) paged optimziers to manage memory 
spikes.
"""

summary = summarize(paper_text)
print('📝 Generated Summary:')
print(summary)

In [ ]:
# Interactive widget (works in Jupyter)
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    text_input = widgets.Textarea(
        placeholder='Paste paper abstract here...',
        layout=widgets.Layout(width='100%', height='200px')
    )
    output_box = widgets.Textarea(
        placeholder='Summary will appear here...',
        layout=widgets.Layout(width='100%', height='120px'),
        disabled=True
    )
    btn = widgets.Button(description='Summarize', button_style='primary')

    def on_click(b):
        output_box.value = '⏳ Generating...'
        output_box.value = summarize(text_input.value)

    btn.on_click(on_click)
    display(HTML('<h3>Interactive Summarizer</h3>'))
    display(text_input, btn, output_box)

except ImportError:
    print('ipywidgets not available — use the summarize() function directly.')

In [ ]:
# Batch demo on test examples from both datasets
from src.data.dataset_loader import load_datasets, load_config

cfg = load_config()
cfg['data']['arxiv_test_size'] = 3
cfg['data']['arxiv_train_size'] = 10
cfg['data']['arxiv_val_size'] = 5

_, _, test_ds = load_datasets(cfg)

for i, example in enumerate(test_ds.select(range(3))):
    print(f'\n--- Example {i+1} ---')
    pred = summarize(example['source'][:1500])
    print(f'Reference : {example["target"][:200]}')
    print(f'Prediction: {pred[:200]}')